<a href="https://colab.research.google.com/github/mhowlin-web/TP_RAG_ARCA/blob/main/08_RAG_HuggingFace_API.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP RAG y Agentes

## RAG sobre trámites de Monotributo en ARCA utilizando un LLM mediante Hugging Face API

En este notebook construyo una versión del sistema RAG en la que el modelo de lenguaje no se ejecuta localmente en Google Colab.

Mantengo la etapa de recuperación de información mediante Pinecone, pero reemplazo el modelo Qwen ejecutado localmente por un modelo de lenguaje accesible mediante la API de Hugging Face.

El objetivo es enviar al modelo únicamente la pregunta del usuario y el contexto recuperado desde los documentos oficiales de ARCA.

El flujo completo que implemento es:

1. Cargar las credenciales.
2. Conectarme con Pinecone.
3. Cargar el modelo de embeddings.
4. Recuperar documentos relevantes.
5. Construir el contexto.
6. Enviar la pregunta y el contexto a Hugging Face mediante API.
7. Obtener la respuesta del LLM.
8. Evaluar el resultado del RAG.

En este notebook el modelo generativo se ejecuta de manera remota. Google Colab solamente realiza las llamadas a las APIs.

## Instalación de librerías

En esta celda instalo las librerías que necesito para conectarme con Pinecone, generar embeddings y utilizar la API de inferencia de Hugging Face.

No necesito instalar `transformers` porque en este notebook no voy a ejecutar el modelo de lenguaje localmente.

In [16]:
!pip install -q pinecone sentence-transformers huggingface_hub

## Importación de librerías

En esta celda importo las librerías que utilizaré para construir el RAG.

Utilizo `InferenceClient` de Hugging Face para enviar las consultas al modelo de lenguaje mediante API.

In [17]:
from pinecone import Pinecone
from sentence_transformers import SentenceTransformer
from huggingface_hub import InferenceClient

from google.colab import userdata

## Carga de credenciales

En esta celda recupero las credenciales almacenadas en los Secrets de Google Colab.

Utilizo una clave para Pinecone y un token de Hugging Face.

No escribo las claves directamente en el notebook para evitar exponer información sensible en GitHub.

In [18]:
PINECONE_API_KEY = userdata.get("PINECONE_API_KEY")
HF_TOKEN = userdata.get("HUGGINGFACE_TOKEN")

if not PINECONE_API_KEY:
    raise ValueError(
        "No se encontró PINECONE_API_KEY en Colab Secrets"
    )

if not HF_TOKEN:
    raise ValueError(
        "No se encontró HUGGINGFACE_TOKEN en Colab Secrets"
    )

print("Credenciales cargadas correctamente.")

Credenciales cargadas correctamente.


## Configuración de Pinecone

En esta celda defino el índice de Pinecone que contiene los embeddings del corpus de ARCA.

Utilizo el índice específico creado para este proyecto y no el índice `tutorial` utilizado durante el aprendizaje.

En este notebook solamente voy a consultar los vectores almacenados. No voy a volver a cargar los embeddings.

In [19]:
PINECONE_INDEX_NAME = "arca-monotributo"

pc = Pinecone(
    api_key=PINECONE_API_KEY
)

index = pc.Index(
    PINECONE_INDEX_NAME
)

print("Conectado a Pinecone.")
print(
    f"Índice utilizado: {PINECONE_INDEX_NAME}"
)

Conectado a Pinecone.
Índice utilizado: arca-monotributo


## Verificación del índice

En esta celda verifico que Pinecone contiene los vectores que cargué en el notebook anterior.

Espero encontrar aproximadamente los 43 embeddings correspondientes a los chunks del corpus.

In [20]:
stats = index.describe_index_stats()

print("=" * 80)
print("ESTADÍSTICAS DE PINECONE")
print("=" * 80)

print(
    f"Total de vectores: "
    f"{stats.total_vector_count}"
)

ESTADÍSTICAS DE PINECONE
Total de vectores: 43


## Carga del modelo de embeddings

En esta celda cargo el mismo modelo de embeddings que utilicé para generar los vectores almacenados en Pinecone.

Es importante utilizar el mismo modelo porque la pregunta debe transformarse al mismo espacio vectorial que los documentos.

In [21]:
EMBEDDING_MODEL_NAME = (
    "sentence-transformers/all-MiniLM-L6-v2"
)

modelo_embeddings = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

print(
    "Modelo de embeddings cargado:"
)

print(
    EMBEDDING_MODEL_NAME
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Modelo de embeddings cargado:
sentence-transformers/all-MiniLM-L6-v2


## Configuración del retrieval

En esta celda defino los parámetros que utilizaré para recuperar información desde Pinecone.

Voy a recuperar los cinco chunks más similares a la pregunta.

Por ahora no aplico un umbral adicional porque primero quiero observar el comportamiento del sistema completo.

In [22]:
TOP_K = 5

## Función de retrieval

En esta celda creo una función que recibe una pregunta, genera su embedding y consulta Pinecone.

Pinecone devuelve los chunks más similares junto con sus metadatos.

In [23]:
def recuperar_documentos(pregunta):

    embedding_pregunta = modelo_embeddings.encode(
        pregunta
    ).tolist()

    resultado = index.query(
        vector=embedding_pregunta,
        top_k=TOP_K,
        include_metadata=True
    )

    return resultado.matches

## Construcción del contexto

En esta celda construyo el contexto que voy a enviar al modelo de lenguaje.

Extraigo el texto de cada chunk recuperado desde Pinecone y los separo mediante una marca.

De esta manera el LLM recibe los documentos recuperados como contexto explícito.

In [24]:
def construir_contexto(matches):

    bloques = []

    for i, match in enumerate(
        matches,
        start=1
    ):

        texto = match.metadata.get(
            "texto",
            ""
        ).strip()

        if texto:

            bloques.append(
                f"[Documento {i}]\n{texto}"
            )

    return "\n\n---\n\n".join(
        bloques
    )

## Conexión con Hugging Face

En esta celda creo el cliente de Hugging Face.

El modelo de lenguaje no se descarga ni se ejecuta en Google Colab.

La consulta será enviada mediante la API de Hugging Face a un proveedor de inferencia remoto.

Utilizo `provider="auto"` para permitir que Hugging Face seleccione automáticamente un proveedor disponible para el modelo.

In [25]:
cliente_hf = InferenceClient(
    api_key=HF_TOKEN,
    provider="auto"
)

print(
    "Cliente de Hugging Face configurado."
)

Cliente de Hugging Face configurado.


## Selección del modelo generativo

En esta celda defino el modelo de lenguaje que voy a utilizar mediante Hugging Face.

No utilizo Qwen localmente como en el notebook 07.

El modelo será ejecutado mediante un proveedor de inferencia remoto.

In [26]:
MODELO_GENERATIVO = "meta-llama/Llama-3.1-8B-Instruct"

print(
    f"Modelo seleccionado: {MODELO_GENERATIVO}"
)

Modelo seleccionado: meta-llama/Llama-3.1-8B-Instruct


## Construcción del prompt

En esta celda construyo las instrucciones que acompañarán al contexto recuperado.

Indico explícitamente al modelo que debe responder utilizando únicamente la información proporcionada.

También le indico que no debe inventar información cuando los documentos recuperados no permitan responder la pregunta.

In [27]:
def construir_mensajes(
    pregunta,
    contexto
):

    system_message = """
Sos un asistente que responde consultas
sobre trámites de ARCA.

Debés utilizar únicamente la información
contenida en el contexto proporcionado.

Reglas:

- Respondé en español.
- No inventes información.
- No utilices información externa al contexto.
- Respondé de manera clara y concisa.
- Conservá correctamente los nombres de trámites,
  servicios y organismos.
- Si el contexto no contiene información suficiente
  para responder, decí exactamente:

"No tengo suficiente información en los documentos recuperados."
"""

    user_message = f"""
CONTEXTO:

{contexto}

PREGUNTA:

{pregunta}
"""

    return [
        {
            "role": "system",
            "content": system_message
        },
        {
            "role": "user",
            "content": user_message
        }
    ]

## Función de generación mediante la API

En esta celda creo la función que envía la pregunta y el contexto a Hugging Face.

A diferencia del notebook 07, acá no utilizo `pipeline()` ni ejecuto el modelo en Colab.

La respuesta es generada remotamente por el proveedor de inferencia seleccionado por Hugging Face.

In [28]:
def generar_respuesta_hf(
    pregunta,
    contexto
):

    mensajes = construir_mensajes(
        pregunta,
        contexto
    )

    respuesta = cliente_hf.chat.completions.create(
        model=MODELO_GENERATIVO,
        messages=mensajes,
        max_tokens=200,
        temperature=0.1
    )

    return (
        respuesta
        .choices[0]
        .message
        .content
        .strip()
    )

## Primera prueba del RAG

En esta celda pruebo el sistema completo con una pregunta concreta sobre la Clave Fiscal.

Primero recupero los documentos desde Pinecone.

Después construyo el contexto.

Finalmente envío la pregunta y el contexto al modelo de lenguaje mediante la API de Hugging Face.

In [32]:
pregunta = "¿Cómo puedo obtener la clave fiscal?"

matches = recuperar_documentos(
    pregunta
)

contexto = construir_contexto(
    matches
)

respuesta = generar_respuesta_hf(
    pregunta,
    contexto
)

print("=" * 80)
print("PREGUNTA")
print("=" * 80)

print(pregunta)

print("\n" + "=" * 80)
print("RESPUESTA RAG")
print("=" * 80)

print(respuesta)

PREGUNTA
¿Cómo puedo obtener la clave fiscal?

RESPUESTA RAG
Puedes obtener la clave fiscal desde la aplicación móvil "ARCA" utilizando la opción "Solicitar o recuperar la clave fiscal". También puedes escanear el código QR para descargarla en tu celular o tablet. Si no puedes obtener la clave fiscal mediante la app, puedes solicitar un turno para presentarte en una dependencia de ARCA.


## Evaluación del RAG

En esta sección pruebo el RAG con diferentes preguntas.

Quiero verificar si el sistema puede:

1. Recuperar información relevante desde Pinecone.
2. Generar respuestas utilizando el contexto recuperado.
3. Responder preguntas sobre diferentes trámites del Monotributo.
4. Reconocer preguntas para las cuales el corpus no contiene información suficiente.

De esta manera puedo evaluar el funcionamiento del RAG antes de considerarlo terminado.

In [33]:
def probar_rag(pregunta):

    # Recuperar documentos desde Pinecone
    matches = recuperar_documentos(
        pregunta
    )

    # Construir contexto
    contexto = construir_contexto(
        matches
    )

    # Generar respuesta mediante Hugging Face
    respuesta = generar_respuesta_hf(
        pregunta,
        contexto
    )

    print("=" * 80)
    print("PREGUNTA")
    print("=" * 80)

    print(pregunta)

    print("\n" + "=" * 80)
    print("RESPUESTA RAG")
    print("=" * 80)

    print(respuesta)

    print("\n" + "=" * 80)
    print("DOCUMENTOS RECUPERADOS")
    print("=" * 80)

    for i, match in enumerate(
        matches,
        start=1
    ):

        print(f"\nDocumento {i}")
        print(f"Score: {match.score:.4f}")

        print(
            f"Documento ID: "
            f"{match.metadata.get('documento_id', 'N/A')}"
        )

        print(
            f"Chunk: "
            f"{match.metadata.get('chunk', 'N/A')}"
        )

    return {
        "pregunta": pregunta,
        "respuesta": respuesta,
        "matches": matches,
        "contexto": contexto
    }

## Prueba 1: obtención de la Clave Fiscal

En esta celda pruebo una pregunta cuya respuesta se encuentra claramente
en el corpus de ARCA.

Espero que Pinecone recupere principalmente fragmentos relacionados con
la Clave Fiscal y que el modelo genere una respuesta basada en esos fragmentos.

In [34]:
resultado_1 = probar_rag(
    "¿Cómo puedo obtener la clave fiscal?"
)

PREGUNTA
¿Cómo puedo obtener la clave fiscal?

RESPUESTA RAG
Puedes obtener la clave fiscal desde la aplicación móvil "ARCA", utilizando la opción "Solicitar o recuperar la clave fiscal". También puedes escanear el código QR para descargarla en tu celular o tablet. Si no puedes obtener la clave fiscal mediante la app, puedes solicitar un turno para presentarte en una dependencia de ARCA.

DOCUMENTOS RECUPERADOS

Documento 1
Score: 0.6147
Documento ID: inicio
Chunk: 1

Documento 2
Score: 0.5985
Documento ID: clave_fiscal
Chunk: 5

Documento 3
Score: 0.5872
Documento ID: tutoriales
Chunk: 40

Documento 4
Score: 0.5835
Documento ID: recategorizacion
Chunk: 24

Documento 5
Score: 0.5828
Documento ID: clave_fiscal
Chunk: 4


## Prueba 2: recategorización

En esta celda pruebo una pregunta sobre la recategorización del Monotributo.

Quiero comprobar que el sistema puede recuperar información relevante
de otra sección del corpus y no solamente información relacionada con
la Clave Fiscal.

In [35]:
resultado_2 = probar_rag(
    "¿Cómo puedo realizar la recategorización del Monotributo?"
)

PREGUNTA
¿Cómo puedo realizar la recategorización del Monotributo?

RESPUESTA RAG
Según el Documento 4, para realizar la recategorización del Monotributo, debes ingresar con clave fiscal al portal Monotributo y realizar la recategorización. Si hubo cambios en alguno de los siguientes parámetros: ingresos, alquileres, superficie afectada a la actividad o energía eléctrica consumida, debes ingresar al portal y realizar la recategorización. Si no se realiza ninguna acción, el sistema entiende que no hubo cambios y el contribuyente permanece en la misma categoría.

DOCUMENTOS RECUPERADOS

Documento 1
Score: 0.7032
Documento ID: recategorizacion
Chunk: 23

Documento 2
Score: 0.6630
Documento ID: recategorizacion
Chunk: 21

Documento 3
Score: 0.6307
Documento ID: recategorizacion
Chunk: 24

Documento 4
Score: 0.6185
Documento ID: recategorizacion
Chunk: 22

Documento 5
Score: 0.5517
Documento ID: recategorizacion
Chunk: 25


## Prueba 3: baja del Monotributo

En esta celda pruebo una consulta relacionada con la baja del Monotributo.

Quiero verificar que el retrieval identifica los fragmentos correspondientes
a este trámite y que el LLM utiliza esa información para elaborar la respuesta.

In [36]:
resultado_3 = probar_rag(
    "¿Cómo puedo dar de baja el Monotributo?"
)

PREGUNTA
¿Cómo puedo dar de baja el Monotributo?

RESPUESTA RAG
Para dar de baja el Monotributo, debes acceder al portal de monotributo y seleccionar la opción "Modificación y baja" en el menú izquierdo. Allí debes seleccionar "Darse de baja del Monotributo" y luego "Dar de baja" (al final de la página). Debes indicar el motivo de la baja y confirmar la operación.

DOCUMENTOS RECUPERADOS

Documento 1
Score: 0.7072
Documento ID: baja
Chunk: 30

Documento 2
Score: 0.6558
Documento ID: baja
Chunk: 29

Documento 3
Score: 0.5411
Documento ID: tutoriales
Chunk: 39

Documento 4
Score: 0.5316
Documento ID: tutoriales
Chunk: 36

Documento 5
Score: 0.4829
Documento ID: tutoriales
Chunk: 37


## Prueba 4: facturación

En esta celda pruebo una pregunta relacionada con la facturación.

El objetivo es comprobar que el RAG puede recuperar información de otra
sección de la documentación oficial de ARCA.

In [37]:
resultado_4 = probar_rag(
    "¿Qué información tiene ARCA sobre la facturación del Monotributo?"
)

PREGUNTA
¿Qué información tiene ARCA sobre la facturación del Monotributo?

RESPUESTA RAG
ARCA proporciona información sobre la facturación del Monotributo a través de varios canales:

1.  El Facturador simplificado: es una herramienta online para emitir tickets de manera segura, que puede utilizarse desde una computadora o un dispositivo móvil.
2.  El Facturador para monotributistas: es una herramienta que permite a los monotributistas emitir comprobantes electrónicos de venta de bienes y servicios desde sus celulares y computadoras.
3.  La guía "¿Cómo utilizo el Facturador?" ofrece instrucciones paso a paso para utilizar el Facturador.
4.  La guía "Recategorización de Monotributo" proporciona información sobre los pasos detallados para recategorizarse.
5.  La guía "Facturación: ¿Cómo utilizo el Facturador?" ofrece información sobre cómo utilizar el Facturador para

DOCUMENTOS RECUPERADOS

Documento 1
Score: 0.5664
Documento ID: facturacion
Chunk: 15

Documento 2
Score: 0.5663
Documen

## Prueba 5: pregunta fuera del corpus

En esta celda realizo una prueba negativa.

La pregunta no está relacionada con los trámites de Monotributo que forman
parte de mi corpus.

Quiero comprobar si el sistema evita utilizar conocimiento externo al contexto
y reconoce que los documentos recuperados no contienen información suficiente.

In [38]:
resultado_5 = probar_rag(
    "¿Quién fue el presidente de Argentina en 1990?"
)

PREGUNTA
¿Quién fue el presidente de Argentina en 1990?

RESPUESTA RAG
No tengo suficiente información en los documentos recuperados para responder a tu pregunta.

DOCUMENTOS RECUPERADOS

Documento 1
Score: 0.3798
Documento ID: clave_fiscal
Chunk: 6

Documento 2
Score: 0.3725
Documento ID: clave_fiscal
Chunk: 8

Documento 3
Score: 0.3592
Documento ID: recategorizacion
Chunk: 25

Documento 4
Score: 0.3481
Documento ID: recategorizacion
Chunk: 26

Documento 5
Score: 0.3371
Documento ID: inicio
Chunk: 3


## Comparación de las pruebas

En esta celda comparo de manera resumida las preguntas y respuestas obtenidas.

Quiero observar si las preguntas que pertenecen al corpus producen respuestas
basadas en la documentación recuperada y si la pregunta que está fuera del
corpus produce una respuesta de falta de información.

In [39]:
resultados = [
    resultado_1,
    resultado_2,
    resultado_3,
    resultado_4,
    resultado_5
]

print("=" * 80)
print("RESUMEN DE LAS PRUEBAS")
print("=" * 80)

for i, resultado in enumerate(
    resultados,
    start=1
):

    print(f"\nPRUEBA {i}")
    print("-" * 80)

    print(
        f"Pregunta: {resultado['pregunta']}"
    )

    print(
        f"Respuesta: {resultado['respuesta']}"
    )

RESUMEN DE LAS PRUEBAS

PRUEBA 1
--------------------------------------------------------------------------------
Pregunta: ¿Cómo puedo obtener la clave fiscal?
Respuesta: Puedes obtener la clave fiscal desde la aplicación móvil "ARCA", utilizando la opción "Solicitar o recuperar la clave fiscal". También puedes escanear el código QR para descargarla en tu celular o tablet. Si no puedes obtener la clave fiscal mediante la app, puedes solicitar un turno para presentarte en una dependencia de ARCA.

PRUEBA 2
--------------------------------------------------------------------------------
Pregunta: ¿Cómo puedo realizar la recategorización del Monotributo?
Respuesta: Según el Documento 4, para realizar la recategorización del Monotributo, debes ingresar con clave fiscal al portal Monotributo y realizar la recategorización. Si hubo cambios en alguno de los siguientes parámetros: ingresos, alquileres, superficie afectada a la actividad o energía eléctrica consumida, debes ingresar al portal 

## Análisis de los resultados

En esta celda analizo los resultados obtenidos en las pruebas anteriores.

Me interesa verificar principalmente:

- si Pinecone recupera documentos relacionados con la pregunta;
- si los fragmentos recuperados contienen información útil;
- si el LLM responde utilizando el contexto;
- si la respuesta es clara y concisa;
- y si el sistema evita responder cuando el corpus no contiene información suficiente.

Si encuentro problemas, puedo modificar posteriormente el retrieval, el tamaño
de los chunks o el prompt.

In [40]:
print("=" * 80)
print("SCORES DE RETRIEVAL")
print("=" * 80)

for i, resultado in enumerate(
    resultados,
    start=1
):

    scores = [
        round(match.score, 4)
        for match in resultado["matches"]
    ]

    print(f"\nPrueba {i}")
    print(f"Pregunta: {resultado['pregunta']}")
    print(f"Scores: {scores}")

SCORES DE RETRIEVAL

Prueba 1
Pregunta: ¿Cómo puedo obtener la clave fiscal?
Scores: [0.6147, 0.5985, 0.5872, 0.5835, 0.5828]

Prueba 2
Pregunta: ¿Cómo puedo realizar la recategorización del Monotributo?
Scores: [0.7032, 0.663, 0.6307, 0.6185, 0.5517]

Prueba 3
Pregunta: ¿Cómo puedo dar de baja el Monotributo?
Scores: [0.7072, 0.6558, 0.5411, 0.5316, 0.4829]

Prueba 4
Pregunta: ¿Qué información tiene ARCA sobre la facturación del Monotributo?
Scores: [0.5664, 0.5663, 0.5516, 0.533, 0.5282]

Prueba 5
Pregunta: ¿Quién fue el presidente de Argentina en 1990?
Scores: [0.3798, 0.3725, 0.3592, 0.3481, 0.3371]
